# 2주차 7교시 · 회원 요약표를 FastAPI로 제공하기

**학생용 복습완성본**

오늘의 질문: **브라우저나 다른 프로그램이 회원 요약을 요청하면 Python은 어떤 데이터로 답할까요?**

오늘의 완성 결과: /health와 /members가 JSON으로 응답하고 /docs에서 직접 요청할 수 있는 서버를 만듭니다.

학생이 막히면 정답 전체를 바로 보여주지 말고, [자료구조] → [입력] → [한 줄씩 읽기] → [정상 결과] 순으로 힌트를 줍니다.


In [ ]:
from pathlib import Path
import pandas as pd

def find_week2_data_dir():
    """노트북을 어느 교시 폴더에서 열어도 2주차 데이터 폴더를 찾습니다."""
    here = Path.cwd().resolve()
    for base in [here, *here.parents]:
        candidates = [
            base / "course_materials" / "matchmaking_data" / "02_week2",
            base / "matchmaking_data" / "02_week2",
            base / "02_week2",
        ]
        for candidate in candidates:
            if (candidate / "activity_batches").exists():
                return candidate
    raise FileNotFoundError("02_week2 데이터 폴더를 찾지 못했습니다. python_lecture 프로젝트 안에서 노트북을 열어 주세요.")

DATA_DIR = find_week2_data_dir()
WEEK2_DIR = DATA_DIR.parents[1] / "week2"
OUTPUT_DIR = WEEK2_DIR / "outputs"
print("데이터 폴더:", DATA_DIR)


## Q1. API가 보낼 결과를 작은 dict로 먼저 만들어 봅니다

**먼저 예상하기**

- API는 화면을 보내는 걸까요, 다른 프로그램이 읽을 데이터를 보내는 걸까요?
- 내 예상: ____________________

**왜 이 일을 하나요?** 화면을 만드는 도구와 데이터를 제공하는 서버의 역할을 구분합니다.

**현재 자료구조** JSON object는 Python dict와 비슷한 이름-값 구조이고 JSON array는 list와 비슷합니다.

**코드 읽기 도움** health 응답은 서버가 살아 있는지와 준비된 행 수를 빠르게 알려 주는 작은 dict입니다.

**정상 결과** {'status': 'ok', 'rows': 1000}

**오류가 나면** 따옴표나 쉼표 SyntaxError. dict의 key와 문자열 값에 따옴표가 있는지 확인합니다.


In [ ]:
example_response = {
    "status": "ok",
    "rows": 1000,
}
print(example_response)

### 강사 설명 메모

- 학생에게 물을 질문: API는 화면을 보내는 걸까요, 다른 프로그램이 읽을 데이터를 보내는 걸까요?
- 기대 답: 다른 프로그램이 읽을 수 있는 이름과 값으로 된 데이터를 보냅니다.
- 그대로 말할 문장: “health 응답은 서버가 살아 있는지와 준비된 행 수를 빠르게 알려 주는 작은 dict입니다.”
- 결과 확인: {'status': 'ok', 'rows': 1000}
- 복구 순서: dict의 key와 문자열 값에 따옴표가 있는지 확인합니다.


## Q2. 서버가 읽을 회원 요약 CSV가 준비되었는지 확인합니다

**먼저 예상하기**

- 서버를 켜기 전에 요약 파일부터 확인해야 하는 이유는 무엇일까요?
- 내 예상: ____________________

**왜 이 일을 하나요?** API 코드와 데이터 준비 코드를 분리해 오류 위치를 쉽게 찾습니다.

**현재 자료구조** summary_path는 Path, summary는 1,000행×16열 DataFrame입니다.

**코드 읽기 도움** API는 요청이 올 때마다 25개 파일을 합치지 않고, 준비된 요약 CSV를 읽어 빠르게 답합니다.

**정상 결과** True와 (1000, 16)

**오류가 나면** FileNotFoundError. prepare_api_data.py를 실행하고 출력된 저장 경로를 확인합니다.


In [ ]:
# 먼저 터미널에서 실행합니다.
# python prepare_api_data.py
summary_path = OUTPUT_DIR / "member_activity_summary.csv"
print(summary_path.exists())
summary = pd.read_csv(summary_path)
print(summary.shape)

### 강사 설명 메모

- 학생에게 물을 질문: 서버를 켜기 전에 요약 파일부터 확인해야 하는 이유는 무엇일까요?
- 기대 답: 요약 파일이 없으면 서버 시작 단계의 read_csv에서 멈추기 때문입니다.
- 그대로 말할 문장: “API는 요청이 올 때마다 25개 파일을 합치지 않고, 준비된 요약 CSV를 읽어 빠르게 답합니다.”
- 결과 확인: True와 (1000, 16)
- 복구 순서: prepare_api_data.py를 실행하고 출력된 저장 경로를 확인합니다.


## Q3. FastAPI 앱을 만들고 상태 확인 주소 /health를 연결합니다

**먼저 예상하기**

- 브라우저가 /health를 요청하면 어떤 함수가 실행되어야 할까요?
- 내 예상: ____________________

**왜 이 일을 하나요?** 서버 프로그램의 가장 작은 정상 상태 확인 주소를 만듭니다.

**현재 자료구조** app은 FastAPI 객체, health 함수의 반환값은 dict이고 FastAPI가 JSON으로 바꿉니다.

**코드 읽기 도움** 데코레이터 @app.get은 웹 주소와 Python 함수를 연결합니다.

**정상 결과** GET /health가 200, JSON은 status ok와 rows 1000

**오류가 나면** NameError: FastAPI 또는 404. import, app 생성, 주소의 첫 슬래시를 확인합니다.


In [ ]:
# api_app_answer.py의 핵심 부분입니다.
from fastapi import FastAPI, Query
app = FastAPI(title="Matchmaking Activity API")

@app.get("/health")
def health():
    return {"status": "ok", "rows": len(summary)}

### 강사 설명 메모

- 학생에게 물을 질문: 브라우저가 /health를 요청하면 어떤 함수가 실행되어야 할까요?
- 기대 답: /health를 @app.get으로 health 함수와 연결합니다.
- 그대로 말할 문장: “데코레이터 @app.get은 웹 주소와 Python 함수를 연결합니다.”
- 결과 확인: GET /health가 200, JSON은 status ok와 rows 1000
- 복구 순서: import, app 생성, 주소의 첫 슬래시를 확인합니다.


## Q4. 요청한 수만큼 회원 요약을 돌려주는 /members를 만듭니다

**먼저 예상하기**

- 한 번에 너무 많은 회원을 요청하지 못하게 하려면 어디에서 제한할까요?
- 내 예상: ____________________

**왜 이 일을 하나요?** 사용자가 요청한 개수만 보내 데이터가 너무 커지는 것을 막습니다.

**현재 자료구조** 반환값은 dict 여러 개가 든 list이며, 각 dict가 회원 한 명의 요약입니다.

**코드 읽기 도움** head(limit)는 앞의 행을 고릅니다. 숫자 결측 NaN은 JSON의 null이 되도록 None으로 바꾸고, orient='records'로 각 행을 dict 하나로 만듭니다.

**정상 결과** /members?limit=3 요청에 회원 dict 3개

**오류가 나면** NameError: Query 또는 NaN JSON 변환 오류. Query import, astype(object)·where 결측 변환, orient='records'를 확인합니다.


In [ ]:
@app.get("/members")
def get_members(limit: int = Query(5, ge=1, le=100)):
    rows = summary.head(limit)
    safe_rows = rows.astype(object).where(pd.notna(rows), None)
    return safe_rows.to_dict(orient="records")

### 강사 설명 메모

- 학생에게 물을 질문: 한 번에 너무 많은 회원을 요청하지 못하게 하려면 어디에서 제한할까요?
- 기대 답: Query의 le=100이 한 번에 요청할 수 있는 최대 개수를 정합니다.
- 그대로 말할 문장: “head(limit)는 앞의 행을 고릅니다. 숫자 결측 NaN은 JSON의 null이 되도록 None으로 바꾸고, orient='records'로 각 행을 dict 하나로 만듭니다.”
- 결과 확인: /members?limit=3 요청에 회원 dict 3개
- 복구 순서: Query import, astype(object)·where 결측 변환, orient='records'를 확인합니다.


## Q5. 서버를 켜고 /docs와 자동 요청으로 응답을 확인합니다

**먼저 예상하기**

- 터미널이 명령을 기다리는 화면으로 돌아오지 않으면 서버가 멈춘 걸까요?
- 내 예상: ____________________

**왜 이 일을 하나요?** Python 파일을 웹 요청을 기다리는 서버 프로세스로 실행합니다.

**현재 자료구조** uvicorn 프로세스는 터미널을 점유하며, 브라우저는 http://127.0.0.1:8000/docs에 접속합니다.

**코드 읽기 도움** status_code 200은 요청이 정상 처리되었다는 뜻이고 json()은 응답 본문을 Python 자료구조로 읽습니다.

**정상 결과** 200 {'status': 'ok', 'rows': 1000}

**오류가 나면** ConnectionError 또는 address already in use. 서버 터미널의 마지막 오류를 보고, 다른 서버를 Ctrl+C로 끄거나 포트를 맞춥니다.


In [ ]:
# 별도 터미널에서 아래 명령을 순서대로 실행합니다.
# 1) api_app_answer.py가 있는 lesson7_fastapi 폴더로 이동합니다.
#    예: cd "압축을 푼 위치/course_materials/week2/lesson7_fastapi"
# 2) 가상환경을 켭니다.
#    conda activate de-course
# 3) 완성된 서버를 시작합니다.
#    python -m uvicorn api_app_answer:app --reload --port 8000

import requests
response = requests.get(
    "http://127.0.0.1:8000/health",
    timeout=5,
)
print(response.status_code, response.json())

### 강사 설명 메모

- 학생에게 물을 질문: 터미널이 명령을 기다리는 화면으로 돌아오지 않으면 서버가 멈춘 걸까요?
- 기대 답: 아니요. 요청을 기다리는 서버가 계속 실행 중인 정상 상태입니다.
- 그대로 말할 문장: “status_code 200은 요청이 정상 처리되었다는 뜻이고 json()은 응답 본문을 Python 자료구조로 읽습니다.”
- 결과 확인: 200 {'status': 'ok', 'rows': 1000}
- 복구 순서: 서버 터미널의 마지막 오류를 보고, 다른 서버를 Ctrl+C로 끄거나 포트를 맞춥니다.


## 마무리 확인

- [ ] 오늘 결과가 **/health와 /members가 JSON으로 응답하고 /docs에서 직접 요청할 수 있는 서버를 만듭니다.**와 같은가?
- [ ] 실행 순서를 자기 말로 설명할 수 있는가?
- [ ] 오류가 났을 때 마지막 줄과 직전 중간 결과를 확인했는가?
